In [ ]:
#| default_exp cli

# SolveIt CLI
> CLI access to the SolveIt API

In [ ]:
#| export
from fastcore.docments import *
from fastcore.script import *
from fastcore.utils import *
from solveit_client.core import *

import inspect,json,sys

In [ ]:
from fastcore.test import *

`_parse_args` splits a `sys.argv`-style list into an operation name, positional args, and keyword args. Boolean flags like `--help` and `--debug` are handled as valueless switches.

In [ ]:
#| export
def _parse_args(a):
    "Extract positional and keyword arguments from `a`=`sys.argv`"
    pos,kw = [],{}
    if len(a)<2: return None,pos,kw
    i=1
    while i<len(a):
        x = a[i]
        if x[:2]=='--':
            k = x[2:]
            if k in ('help','debug'): y = 1
            else:
                i += 1
                y = a[i]
            kw[k] = y
        else: pos.append(a[i])
        i += 1
    a = pos.pop(0) if pos else None
    return a,pos,kw

In [ ]:
test_eq(_parse_args(['sic']), (None, [], {}))
test_eq(_parse_args(['sic', '--help']), (None, [], {'help': 1}))
test_eq(_parse_args(['sic', 'dialog', '--help']), ('dialog', [], {'help': 1}))
test_eq(_parse_args(['sic', 'dialog.add_msg', '--help']), ('dialog.add_msg', [], {'help': 1}))
test_eq(_parse_args(['sic', 'dialog.add_msg', 'new content', '--msg_type', 'note']),
        ('dialog.add_msg', ['new content'], {'msg_type': 'note'}))

`_ns_map` maps CLI namespace strings to their corresponding classes. `_to_json` serializes response objects to JSON using allowlisted keys — `Dialog` returns `name` and `mode`, `Message` returns `id`, `msg_type`, `content`, and `output`.

In [ ]:
#| export
_ns_map = {'client': SolveItClient, 'dialog': Dialog, 'message': Message}

def _pick(d, ks): return {k: d[k] for k in ks if k in d}
msg_keys = ['id','msg_type','content','output']
def _to_json(res):
    if isinstance(res, Dialog): res = _pick(res.data, ['name','mode'])
    elif isinstance(res, Messages): res = [_pick(m.data, msg_keys) for m in res]
    elif isinstance(res, MsgDiff): res = {**_pick(res.msg.data, msg_keys), 'diff': res.diff}
    elif hasattr(res, 'data'): res = _pick(res.data, msg_keys)
    return json.dumps(res, default=str)

`_resolve` takes an operation string like `"dialog.add_msg"` and keyword args, constructs the appropriate object chain (`SolveItClient` → `Dialog` → `Message`), and returns the resolved method plus remaining kwargs. Infrastructure args (`url`, `token`, `name`, `id`) are popped and used for object construction, falling back to env vars (`SOLVEIT_URL`, `SOLVEIT_TOKEN`, `SOLVEIT_DIALOG`).

In [ ]:
#| export
def _resolve(op, kws):
    parts = op.split('.')
    nm = parts[0]
    mthd = parts[1] if len(parts) > 1 else None
    if not mthd: return _ns_map[nm], kws
    elif mthd and 'help' in kws:
        cls = _ns_map[nm]
        if mthd not in dir(cls): raise AttributeError(f"'{nm}' has no method '{mthd}'")
        return getattr(cls, mthd), kws
    sic = SolveItClient(kws.pop('url', None), kws.pop('token', None))
    if nm=='client': obj = sic
    else:
        dlg = Dialog(kws.pop('name', os.environ.get('SOLVEIT_DIALOG')), sic)
        if   nm=='dialog': obj = dlg
        elif nm=='message': obj= Message(kws.pop('id'), dlg)
        else: raise ValueError(f'Unknown object type {nm}')
    if mthd not in dir(obj): raise AttributeError(f"'{nm}' has no method '{mthd}'")
    attr = getattr(type(obj), mthd, None) if 'help' in kws else getattr(obj, mthd)
    return attr, kws

In [ ]:
method, clean_kw = _resolve('client.create_dialog', {'url': 'http://localhost:6001', 'name': 'folder/test'})
test_eq(clean_kw, {'name': 'folder/test'})
test_eq(method.__name__, 'create_dialog')

`sic_cli` is the main entry point that ties parsing, resolution, and output together. It handles three levels of help display: top-level namespace listing, per-namespace method listing (via `inspect.getmembers`), and per-method signature display (via `docments`).

In [ ]:
#| export
def sic_cli(a):
    op, pos, kw = _parse_args(a)
    if not op: return 'Usage: sic <operation> [args]\n\n' + '\n'.join([ f" {nm:10s} {cls.__doc__}"
                                                                        for nm, cls in _ns_map.items()])
    obj, kw = _resolve(op, kw)
    if obj in _ns_map.values():
        mems = L(inspect.getmembers(obj, lambda x: callable(x) or isinstance(x, property))).filter(lambda x: not x[0].startswith('_'))
        lines = [f"  {name:15s} {func.__doc__ or ''}" for name, func in mems]
        return f'Usage: sic {op}.<method> [args]\n\n' + '\n'.join(lines)
    if 'help' in kw:
        kw.pop('help')
        doc = obj.__doc__ or ''
        lines = [doc, '']
        if callable(obj):
            for k, v in docments(obj).items():
                if k in ('self', 'return'): continue
                lines.append(f"  --{k:20s} {v or ''}")
        return '\n'.join(lines)
    res = obj(*pos, **kw) if callable(obj) else obj
    return _to_json(res)

## Tests

In [ ]:
res = sic_cli(['sic'])
assert 'client' in res and SolveItClient.__doc__ in res
assert 'dialog' in res and Dialog.__doc__ in res
assert 'message' in res and Message.__doc__ in res
print(res)

Usage: sic <operation> [args]

 client     SolveIt API client
 dialog     Dialog operations
 message    Message operations


In [ ]:
res = sic_cli(['sic', 'dialog', '--help'])
assert 'add_msg' in res
assert 'messages' in res
assert 'find_msgs' in res
assert Dialog.__doc__ not in res

res = sic_cli(['sic', 'client', '--help'])
assert 'create_dialog' in res

res = sic_cli(['sic', 'message', '--help'])
assert 'exec' in res
assert 'update' in res
print(res)

Usage: sic message.<method> [args]

  del_lines       Delete lines from `start_line` to `end_line`.
  delete          Delete this message from its dialog.
  exec            Execute this message and poll until completion or timeout.
  insert_line     Insert `new_str` at line number `insert_line`.
  link            Link to the message.
  num_content     Return content with line numbers.
  replace_lines   Replace lines from `start_line` to `end_line` with `new_content`.
  str_replace     Replace `old_str` with `new_str` in message content (must match exactly once).
  strs_replace    Replace each string in `old_strs` with corresponding string in `new_strs`.
  update          Update message fields and return a `MsgDiff` with the change.


In [ ]:
res = sic_cli(['sic', 'dialog.messages', '--help'])
assert 'All messages' in res

res = sic_cli(['sic', 'dialog.add_msg', '--help'])
for arg, doc in docments(Dialog.add_msg).items():
    if arg in ('self', 'return'): continue
    test(arg, res, operator.in_)

res = sic_cli(['sic', 'dialog.to_xml', '--help'])
for arg, doc in docments(Dialog.to_xml).items():
    if arg in ('self', 'return'): continue
    test(arg, res, operator.in_)

print(res)

Return dialog messages as an XML string.

  --msg_type             optional limit by message type ('code', 'note', or 'prompt')
  --nums                 Whether to show line numbers
  --include_output       Include output in returned dict?
  --trunc_out            Middle-out truncate code output to 100 characters (only applies if `include_output`)?
  --trunc_in             Middle-out truncate cell content to 80 characters?


In [ ]:
# --- setup ---
sic_cli(['sic', 'client.create_dialog', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2'])

# --- dialog.add_msg ---
res = json.loads(sic_cli(['sic', 'dialog.add_msg', 'hello world', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2', '--msg_type', 'note']))
test_eq(res['msg_type'], 'note')
test_eq(res['content'], 'hello world')
assert 'id' in res

# --- dialog.messages ---
res = json.loads(sic_cli(['sic', 'dialog.messages', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2']))
assert isinstance(res, list)
assert len(res) > 0

# --- message.update ---
msg_id = res[0]['id']
res = json.loads(sic_cli(['sic', 'message.update', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2', '--id', msg_id, '--content', 'updated']))
test_eq(res['content'], 'updated')

# --- message.exec (code cell) ---
code_msg = json.loads(sic_cli(['sic', 'dialog.add_msg', '1+1', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2']))
res = json.loads(sic_cli(['sic', 'message.exec', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2', '--id', code_msg['id']]))
assert 'output' in res

# --- edge: unknown namespace ---
try:
    sic_cli(['sic', 'foo.bar', '--url', 'http://localhost:6001'])
    assert False, "Should have raised"
except ValueError: pass

# --- edge: unknown method ---
try:
    sic_cli(['sic', 'dialog.nonexistent', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2'])
    assert False, "Should have raised"
except AttributeError: pass

# --- cleanup ---
sic_cli(['sic', 'dialog.delete', '--url', 'http://localhost:6001', '--name', 'tmp/cli_test2'])

'{"success": "deleted \\"/home/natedawg/tmp/cli_test2\\""}'

`sic` is the CLI entry point — a thin wrapper that passes `sys.argv` to `sic_cli` and prints the result. Wire it up as a console script in `pyproject.toml`:

```toml
[project.scripts]
sic = "solveit_client.cli:sic"
```

In [ ]:
#| export
def sic():
    "SolveIt CLI"
    print(sic_cli(sys.argv))

In [ ]:
!sic

Usage: sic <operation> [args]

 client     SolveIt API client
 dialog     Dialog operations
 message    Message operations


# fin

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

/home/natedawg/aai-ws/nbdev/nbdev/export.py:55: UserWarning: Notebook '/home/natedawg/aai-ws/solveit_client/nbs/01_cli_dup1.ipynb' uses `#| export` without `#| default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev-migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#| export` without `#| default_exp` cell.\n"
